# PoolCue Vision Assist — YOLOv11 Training
**AIPI 590 Final Project**

### Running this notebook
- **VSCode + remote server (recommended):** Open this file in VSCode, select the remote kernel, set `MODE = 'local'` in Step 1, update `ZIP_PATH` to the absolute path of the zip on the server
- **Google Colab:** Upload zip to Google Drive, set `MODE = 'colab'`, update `ZIP_PATH`

### After training
Download `best.pt` from `pool_vision/yolo11n_pool/weights/best.pt` → place on Pi at `models/pool_vision/best.pt`

In [ ]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
DATASET_DIR = '/tmp/pool_dataset'
OUTPUT_DIR  = './pool_vision'
EPOCHS      = 50
BATCH       = 32
IMGSZ       = 320
DEVICE      = 0    # T4 GPU
# ────────────────────────────────────────────────────────────────────────────

In [ ]:
# Step 1: Install dependencies
!pip install ultralytics roboflow -q

In [ ]:
# Step 2: Download dataset from Roboflow
from roboflow import Roboflow

# Get your free API key at roboflow.com — paste it here
rf = Roboflow(api_key="YOUR_ROBOFLOW_API_KEY")
project = rf.workspace("nidacorian-protonmail-com").project("pool-billiard")
version = project.version(1)
dataset = version.download("yolov11", location=DATASET_DIR)

print("Downloaded to:", dataset.location)

In [ ]:
# Step 3: Verify split
import os
for split in ['train', 'valid', 'test']:
    path = os.path.join(DATASET_DIR, split, 'images')
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"{split}: {count} images")

In [ ]:
# Step 4: Write data.yaml with correct Colab paths
import yaml, os

data_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')

data_cfg = {
    'train': f'{DATASET_DIR}/train/images',
    'val':   f'{DATASET_DIR}/valid/images',
    'test':  f'{DATASET_DIR}/test/images',
    'nc': 16,
    'names': ['0','1','2','3','4','5','6','7','8','9','10','11','12','13','14','15']
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_cfg, f)

print("data.yaml written:")
print(open(data_yaml_path).read())

In [ ]:
# Step 5: Train YOLOv11 nano
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=20,
    project=OUTPUT_DIR,
    name='yolo11n_pool',
    device=DEVICE,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
)

BEST_PT = f'{OUTPUT_DIR}/yolo11n_pool/weights/best.pt'
print(f'\nBest weights saved to: {BEST_PT}')

In [ ]:
# Step 6: Train YOLOv11 nano
from ultralytics import YOLO

model = YOLO('yolo11n.pt')

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=20,
    project=OUTPUT_DIR,
    name='yolo11n_pool',
    device=DEVICE,
    augment=True,
    hsv_h=0.015,
    hsv_s=0.4,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=1.0,
)

BEST_PT = f'{OUTPUT_DIR}/yolo11n_pool/weights/best.pt'
print(f'\nBest weights: {BEST_PT}')

In [ ]:
# Step 7: Evaluate
best = YOLO(BEST_PT)
metrics = best.val(data=data_yaml_path)

print(f"mAP50:     {metrics.box.map50:.3f}")
print(f"mAP50-95:  {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
# Step 8: Print class names — verify order matches detector.py
print('Class names in trained model:')
for idx, name in best.names.items():
    print(f'  {idx}: {name}')

In [ ]:
# Step 9: Export to ONNX for Pi
best.export(format='onnx', imgsz=640, simplify=True, opset=12)
BEST_ONNX = BEST_PT.replace('.pt', '.onnx')
print(f'ONNX saved: {BEST_ONNX}')
print(f'\nCopy {BEST_PT} → models/pool_vision/best.pt on your Pi')

In [ ]:
# Step 10: Download (Colab only)
if MODE == 'colab':
    from google.colab import files
    files.download(BEST_PT)
    files.download(BEST_ONNX)
else:
    print(f'Local mode — copy weights manually:')
    print(f'  {BEST_PT}')
    print(f'  {BEST_ONNX}')